In [2]:
import os
import glob
import polars as pl
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Feature Data Directory (Where the CSVs to be updated are located)

# Change the Drive letter to "F" or "D" depending on the mounted SSD drive in the PC

#======================================================================================================================
# Select this path  for training data updates
#FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Dust2")

# Select this path  for testing data
FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Dust2")

# Raw Data Directory (To extract the 'round_end' ground truth)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Standardized Team Mapping
TEAM_MAP = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}

# ==========================================
# 2. DATA RETRIEVAL & UPDATING LOGIC
# ==========================================
def update_features_with_targets():
    """Iterates through feature CSVs, retrieves ground-truth labels, and appends them."""
    
    if not FEAT_DIR.exists():
        print(f"Error: Feature directory does not exist -> {FEAT_DIR}")
        return

    csv_files = list(FEAT_DIR.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in -> {FEAT_DIR}")
        return

    print(f"Found {len(csv_files)} matches to update. Processing...\n")
    updated_count = 0

    for filepath in csv_files:
        match_id = filepath.stem
        
        # --- A. Find the Raw Round_End Data ---
        match_dirs = list(RAW_DATA_DIR.rglob(match_id))
        if not match_dirs:
            print(f"Skipping {match_id}: Raw data directory not found.")
            continue
            
        re_files = glob.glob(str(match_dirs[0] / "round_end*"))
        if not re_files:
            print(f"Skipping {match_id}: 'round_end' channel missing.")
            continue

        # --- B. Extract the Target Labels ---
        try:
            df_re = pl.read_parquet(re_files[0])
            
            # Map the winner codes and keep only 'round' and 'target'
            df_labels = df_re.with_columns(
                pl.col("winner_team_code").cast(pl.Utf8).replace(TEAM_MAP).alias("target")
            ).select(["round", "target"])
            
        except Exception as e:
            print(f"Error reading 'round_end' for {match_id}: {e}")
            continue

        # --- C. Update the Feature Data ---
        try:
            df_feat = pl.read_csv(filepath)
            
            # Safety check: If "target" already exists from a previous run, drop it to prevent duplicates like "target_right"
            if "target" in df_feat.columns:
                df_feat = df_feat.drop("target")
                
            # Perform a Left Join on the "round" column
            # This ensures every round in the feature data gets its corresponding label
            df_updated = df_feat.join(df_labels, on="round", how="left")
            
            # --- D. Save the Changes (Overwrite) ---
            df_updated.write_csv(filepath)
            updated_count += 1
            
            print(f"Successfully updated: {match_id}")
            
        except Exception as e:
            print(f"Error updating feature file {match_id}: {e}")

    print("\n" + "="*40)
    print(f" UPDATE COMPLETE: {updated_count}/{len(csv_files)} files modified.")
    print("="*40)

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Note: Because this script overwrites data, ensure no other processes are locking the CSV files.
    update_features_with_targets()

Found 21 matches to update. Processing...

Successfully updated: 0706d00b-ff59-4a78-900f-4c2ffecf2d14
Successfully updated: 23aaf423-615d-45c4-82a0-837a782d831f
Successfully updated: 7d461d04-d772-4acb-8137-e2893686b35c
Successfully updated: bb03f650-6639-466f-af7e-f88fc1cb6b2d
Successfully updated: a52206c8-a339-461a-b3d6-f4a9b8ec49c5
Successfully updated: 756c0d58-0b19-485c-bac8-da4de590679f
Successfully updated: ae6d0b9a-0b59-4847-afe8-ed7a391c79b0
Successfully updated: 5a8f1a0f-732c-47a1-a8e1-cd87c21595c3
Successfully updated: c41d54dd-4109-441d-9f35-6e3af7dc20e4
Successfully updated: 102860e5-87ed-4b47-a129-27ba0b39267f
Successfully updated: e5881b09-b922-43a0-a65b-122b1a409c75
Successfully updated: cbcc63cc-7f08-4074-8216-8d4838ac5ffd
Successfully updated: f2ca02f8-8d31-47e7-aef6-7072ce6efdb0
Successfully updated: 364ad321-19a9-494b-887b-dd6027b27d63
Successfully updated: 825c4d01-12fa-482b-aea9-c6f05e5a0eae
Successfully updated: 63445000-21ea-4f5e-84ac-f19fa8d81743
Successfully 